<a href="https://colab.research.google.com/github/LtFordo/SCY1101_Precios_Consumidor/blob/main/ProyectoProgramaci%C3%B3nCienciaDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/LtFordo/SCY1101_Precios_Consumidor

Cloning into 'SCY1101_Precios_Consumidor'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [122]:
# Importación de las librerías a utilizar.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Se carga el dataset en su estado original para ejecutar posteriormente procesos de limpieza,
#validación y transformación de datos.
raw=pd.read_csv('SCY1101_Precios_Consumidor/data/precio_consumidor_2025_raw.csv')

# Respaldo del dataset original para trabajar sin reparos.
df = raw.copy()

In [129]:
try:
    print("--- Auditoría Inicial de Integridad ---")
    # Revisa tipos de datos y nulos de forma general [7]
    print(df.info())
    # Suma total de nulos por cada columna para dimensionar el Bias [7, 11]
    print("\nNulos iniciales:\n", df.isnull().sum())
    # Muestra Min/Max para detectar valores imposibles u outliers [7]
    print("\nEstadísticas descriptivas:\n", df.describe())

except Exception as e:
    print(f"Error en la fase de auditoría: {e}")
df['Region'].value_counts()

--- Auditoría Inicial de Integridad ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366618 entries, 0 to 366617
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Anio                     348367 non-null  float64
 1   Mes                      348716 non-null  float64
 2   Semana                   348563 non-null  float64
 3   Fecha inicio             348628 non-null  object 
 4   Fecha termino            348658 non-null  object 
 5   ID region                348217 non-null  float64
 6   Region                   348480 non-null  object 
 7   Sector                   348682 non-null  object 
 8   Tipo de punto monitoreo  348723 non-null  object 
 9   Grupo                    348650 non-null  object 
 10  Producto                 348662 non-null  object 
 11  Unidad                   348436 non-null  object 
 12  Precio minimo            348938 non-null  object 
 13  Precio maximo      

,count
Region,
Región Metropolitana de Santiago,69114
Región de La Araucanía,62762
Región de Valparaíso,58117
Región de Coquimbo,29576
Región del Biobío,28119
Región de Ñuble,27001
Región del Maule,25877
Región de Los Lagos,24960
Región de Arica y Parinacota,15622


In [120]:
try:
    # 1. Eliminación de duplicados para evitar el sobreajuste (Overfitting) [7, 12]
    df.drop_duplicates(inplace=True)

    # 2. Corrección de inconsistencia de tipos [6, 14]
    # Forzamos conversión a numérico para asegurar cálculos vectoriales rápidos [15, 16]
    cols_precios = ['Precio minimo', 'Precio maximo', 'Precio promedio']
    for col in cols_precios:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Conversión de fechas a tipo datetime para ingeniería de atributos [14]
    df['Fecha inicio'] = pd.to_datetime(df['Fecha inicio'], errors='coerce')
    df['Fecha termino'] = pd.to_datetime(df['Fecha termino'], errors='coerce')

    # 3. Tratamiento de Outliers (Valores Atípicos) [6, 17]
    # Filtro lógico: Meses y semanas deben ser cronológicamente válidos [18]
    df = df[(df['Mes'] <= 12) & (df['Semana'] <= 53)]
    # Eliminación de precios negativos (errores de medida) [17]
    df = df[df['Precio promedio'] >= 0]

    # 4. Gestión de Varianza: Eliminación de columnas con 1 sola categoría (Varianza nula) [19]
    # Justificación: No aportan información ni ayudan a encontrar patrones [19]
    for col in df.columns:
        if df[col].nunique() <= 1:
            df.drop(columns=[col], inplace=True)
            print(f"Columna eliminada por varianza nula: {col}")

    # 5. Tratamiento de Missing Values (Imputación profesional) [17, 18]
    # Se imputan categorías con etiquetas y precios con la media para no perder info [17]
    df[['Sector', 'Grupo', 'Unidad']] = df[['Sector', 'Grupo', 'Unidad']].fillna('No especificado')
    df['Precio promedio'] = df['Precio promedio'].fillna(df['Precio promedio'].mean())

    # Barrido final para asegurar 0 nulos (Requisito de integridad) [8]
    df.dropna(inplace=True)

    print("\nFase de limpieza completada exitosamente.")

except Exception as e:
    print(f"Error en la fase de limpieza: {e}")
df



Fase de limpieza completada exitosamente.


,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio
12,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Carnicería,Carne bovina,Palanca,$/kilo,12398.0,12998.0,12698.000000
123,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Supermercado,Abarrotes y otros,Spaghetti N°5,$/envase 400 gramos,520.0,1330.0,1015.384615
211,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Feria libre,Frutas,Limón|Sin especificar|2a amarillo,$/kilo,700.0,700.0,700.000000
304,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Carne bovina,Posta Negra,$/kilo,9690.0,10490.0,10090.000000
354,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Hortalizas,Tomate|Larga vida|Primera,$/kilo,1950.0,1990.0,1963.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366311,11.0,47.0,2025-11-17,2025-11-21,10.0,Región de Los Lagos,Llanquihue,Supermercado,Abarrotes y otros,Quinoa,$/envase 1 kilo,7375.0,11800.0,9716.666666
366327,3.0,11.0,2025-03-10,2025-03-14,5.0,Región de Valparaíso,Viña del Mar,Supermercado,Carne de Cerdo - Ave - Cordero,Chuleta (Parrillera),$/kilo,4990.0,5790.0,5440.000000
366334,12.0,50.0,2025-12-08,2025-12-12,5.0,Región de Valparaíso,Quillota,Supermercado,Carne bovina,Posta Negra,$/kilo,9990.0,11490.0,10923.333333
366358,2.0,7.0,2025-02-10,2025-02-14,13.0,Región Metropolitana de Santiago,Oriente,Supermercado,Carne bovina,Lomo Vetado,No especificado,14990.0,17590.0,16061.428571


In [121]:
try:
    # 1. Ingeniería de Características: Extraer atributos numéricos de las fechas [21, 22]
    df_transformado = df.copy()
    df_transformado['Mes_Registro'] = df_transformado['Fecha termino'].dt.month
    df_transformado['Dia_Semana'] = df_transformado['Fecha termino'].dt.dayofweek

    # 2. Codificación Categórica (One-Hot Encoding con 1 y 0) [14, 22]
    # Se transforman regiones y categorías en variables binarias puras (int)
    cols_cat = ['Region', 'Sector', 'Grupo']
    df_transformado = pd.get_dummies(df_transformado, columns=cols_cat, prefix=cols_cat)

    # Conversión explícita de booleanos a enteros (1 y 0) para optimizar memoria [16, 23]
    cols_bool = df_transformado.select_dtypes(include=['bool']).columns
    df_transformado[cols_bool] = df_transformado[cols_bool].astype(int)

    # 3. Estandarización de Variables Numéricas [14, 22]
    # Ajustamos los valores para que tengan una escala similar (media 0, desv. 1) [22, 24]
    scaler = StandardScaler()
    df_transformado['Precio_Promedio_Escalado'] = scaler.fit_transform(df_transformado[['Precio promedio']])

    # 4. Eliminación de columnas de texto remanentes [21, 25]
    df_final = df_transformado.select_dtypes(include=['number'])

    print("Fase de transformación avanzada finalizada.")
    print(f"Tipos de datos únicos en el dataset final: {df_final.dtypes.unique()}")

except Exception as e:
    print(f"Error en la fase de transformación: {e}")
df

Fase de transformación avanzada finalizada.
Tipos de datos únicos en el dataset final: [dtype('float64') dtype('int32') dtype('int64')]


/tmp/ipykernel_28071/983716795.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transformado['Precio_Promedio_Escalado'] = scaler.fit_transform(df_transformado[['Precio promedio']])


,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio
12,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Carnicería,Carne bovina,Palanca,$/kilo,12398.0,12998.0,12698.000000
123,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Supermercado,Abarrotes y otros,Spaghetti N°5,$/envase 400 gramos,520.0,1330.0,1015.384615
211,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Feria libre,Frutas,Limón|Sin especificar|2a amarillo,$/kilo,700.0,700.0,700.000000
304,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Carne bovina,Posta Negra,$/kilo,9690.0,10490.0,10090.000000
354,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Hortalizas,Tomate|Larga vida|Primera,$/kilo,1950.0,1990.0,1963.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366311,11.0,47.0,2025-11-17,2025-11-21,10.0,Región de Los Lagos,Llanquihue,Supermercado,Abarrotes y otros,Quinoa,$/envase 1 kilo,7375.0,11800.0,9716.666666
366327,3.0,11.0,2025-03-10,2025-03-14,5.0,Región de Valparaíso,Viña del Mar,Supermercado,Carne de Cerdo - Ave - Cordero,Chuleta (Parrillera),$/kilo,4990.0,5790.0,5440.000000
366334,12.0,50.0,2025-12-08,2025-12-12,5.0,Región de Valparaíso,Quillota,Supermercado,Carne bovina,Posta Negra,$/kilo,9990.0,11490.0,10923.333333
366358,2.0,7.0,2025-02-10,2025-02-14,13.0,Región Metropolitana de Santiago,Oriente,Supermercado,Carne bovina,Lomo Vetado,No especificado,14990.0,17590.0,16061.428571


In [116]:
# =================================================================
# FASE 3: TRANSFORMACIÓN AVANZADA (FEATURE ENGINEERING & ENCODING)
# =================================================================
# Justificación: Se transforman categorías a números (Encoding) y se escalan
# las variables para optimizar el rendimiento de los algoritmos [17-19].

try:
    # 1. Tratamiento de Fechas (Feature Engineering)
    # Justificación: Las fechas como 'datetime' no son procesables directamente.
    # Extraemos atributos numéricos que aportan valor predictivo [2, 3].
    for col in ['Fecha inicio', 'Fecha termino']:
        df[f'{col}_dia'] = df[col].dt.day
        df[f'{col}_mes'] = df[col].dt.month
        df[f'{col}_dia_semana'] = df[col].dt.dayofweek

    # Eliminamos las columnas de fecha originales (objetos no numéricos)
    df.drop(columns=['Fecha inicio', 'Fecha termino'], inplace=True)

    # 2. Codificación Categórica Completa (One-Hot Encoding)
    # Justificación: Transformamos el texto (Sector, Grupo, Producto, Unidad)
    # en variables binarias (0 y 1) para eliminar el tipo 'object' [3, 8].
    cols_a_codificar = ['Sector', 'Tipo de punto monitoreo', 'Grupo', 'Producto', 'Unidad']
    df_final = pd.get_dummies(df, columns=cols_a_codificar, prefix=cols_a_codificar)

    # 3. Conversión de Booleanos a Enteros (0 y 1)
    # Justificación: Para asegurar que el ndarray resultante sea puramente numérico
    # y evitar ambigüedades en la memoria computacional [6].
    cols_bool = df_final.select_dtypes(include=['bool']).columns
    df_final[cols_bool] = df_final[cols_bool].astype(int)

    # 4. Estandarización Final
    # Justificación: Una vez que todo es numérico, escalamos para que las variables
    # tengan una magnitud similar, facilitando la convergencia del modelo [3, 9].
    scaler = StandardScaler()
    # Escalamos todas las columnas excepto los IDs y variables binarias ya codificadas
    # (Opcional: puedes escalar todo el set si el modelo lo requiere)

    print("--- TRANSFORMACIÓN A DATASET 100% NUMÉRICO COMPLETADA ---")
    print(df_final.info())
    print(f"\nTipos de datos únicos en el dataset: {df_final.dtypes.unique()}")
    print(f"¿Existen nulos?: {df_final.isnull().sum().sum()}")

except Exception as e:
    # El manejo de excepciones garantiza la robustez profesional exigida [10].
    print(f"Error crítico en la transformación numérica: {e}")
df_final['Region'].value_counts()

--- TRANSFORMACIÓN A DATASET 100% NUMÉRICO COMPLETADA ---
<class 'pandas.core.frame.DataFrame'>
Index: 5892 entries, 12 to 366616
Columns: 495 entries, Anio to Unidad_No especificado
dtypes: float64(13), int64(481), object(1)
memory usage: 22.3+ MB
None

Tipos de datos únicos en el dataset: [dtype('float64') dtype('O') dtype('int64')]
¿Existen nulos?: 3366


,count
Region,
Región Metropolitana de Santiago,1108
Región de La Araucanía,961
Región de Valparaíso,947
Región de Coquimbo,457
Región del Biobío,447
Región del Maule,438
Región de Ñuble,424
Región de Los Lagos,413
Región de Arica y Parinacota,250


In [ ]:
try:
    # 1. Eliminación de Duplicados: Evita sesgar las estadísticas del modelo [6, 9]
    df.drop_duplicates(inplace=True)

    # 2. Corrección de tipos: Conversión a numérico y datetime [10]
    # Usamos errors='coerce' para gestionar datos mal formateados como NaN
    cols_precios = ['Precio minimo', 'Precio maximo', 'Precio promedio']
    for col in cols_precios:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Fecha inicio'] = pd.to_datetime(df['Fecha inicio'], errors='coerce')
    df['Fecha termino'] = pd.to_datetime(df['Fecha termino'], errors='coerce')

    print("Corrección de formatos y eliminación de duplicados finalizada.")
except Exception as e:
    print(f"Error en corrección de tipos: {e}")